In [ ]:
!pip install langchain langchain-community langchain-core
!pip install pinecone-client
!pip install python-dotenv
!pip install sentence-transformers
!pip install chromadb
!pip install pypdf


In [ ]:
import sys
print(sys.executable)


In [ ]:
import langchain
import chromadb
import pinecone
from sentence_transformers import SentenceTransformer

print("LangChain OK")
print("ChromaDB OK")
print("Pinecone Client OK")
print("Sentence-Transformers OK")


In [ ]:
!pip uninstall -y pinecone-client


In [ ]:
!pip install pinecone


In [ ]:
import langchain
import chromadb
import pinecone
from sentence_transformers import SentenceTransformer

print("LangChain OK")
print("ChromaDB OK")
print("Pinecone Client OK")
print("Sentence-Transformers OK")


In [ ]:
import os

folder = r"E:\RAG Project\docs"
print(os.listdir(folder))


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = r"E:\RAG Project\docs\Computer hacking.pdf"
loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("Total pages loaded:", len(documents))
print(documents[0].page_content[:500])


In [ ]:
for i, page in enumerate(documents):
    print(f"PAGE {i}")
    print(page.page_content[:400])  # show first 400 chars
    print("-" * 60)


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Extract all page text into one big string
full_text = "\n".join([page.page_content for page in documents])

# Initialize chunker
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # max size of each chunk
    chunk_overlap=100,   # repeated text to maintain context
    separators=["\n\n", "\n", ".", " ", ""]
)

# Create chunks
chunks = text_splitter.split_text(full_text)

print("Total chunks created:", len(chunks))
print("\nSample chunk:\n")
print(chunks[0][:500])


In [ ]:
print(chunks[0][:100])
print("----")
print(chunks[0][:200])
print("----")
print(chunks[0][:300])
print("----")
print(chunks[0][:500])
print("----")
print(chunks[0][:800])


# Embedding 

In [ ]:
from sentence_transformers import SentenceTransformer

# Load embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for each chunk
embeddings = embedding_model.encode(chunks)

print("Total embeddings created:", len(embeddings))
print("Embedding vector size:", len(embeddings[0]))
print("\nSample embedding vector:\n")
print(embeddings[0][:10])  # show first 10 numbers


# Storing the embeddings into chromaDB

In [ ]:
import chromadb

# Create a persistent Chroma client (new API)
client = chromadb.PersistentClient(path="E:\\RAG Project\\chroma_db")

# Create or load a collection
collection = client.get_or_create_collection(
    name="hacking_knowledge",
    metadata={"hnsw:space": "cosine"}   # cosine similarity
)

# Insert all chunks + embeddings
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=embeddings
)

print("Chroma DB created and data inserted successfully!")


In [ ]:
def search(query, top_k=3):
    # Embed the query
    query_embedding = embedding_model.encode([query])[0]

    # Search in Chroma
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    return results


In [ ]:
result = search("What is hacking?")
print(result)


In [ ]:
# Run this cell in Jupyter
!pip install openai --quiet


In [ ]:
# Run this cell in Jupyter
import os
from dotenv import load_dotenv
import openai

load_dotenv()  # loads OPENAI_API_KEY from E:\RAG Project\.env
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env — please add it and re-run this cell")

openai.api_key = OPENAI_API_KEY

print("OpenAI key loaded, length:", len(OPENAI_API_KEY))


# Loading the env file 

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

OPENAI_KEY = os.getenv("OPENAI_API_KEY")

print("OpenAI key:", OPENAI_KEY)


In [ ]:
import os

if os.path.exists(".env.txt"):
    os.rename(".env.txt", ".env")
    print("Renamed to .env")
else:
    print("No .env.txt found")


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print("OPENAI KEY =", os.getenv("OPENAI_API_KEY"))
print("PINECONE KEY =", os.getenv("PINECONE_API_KEY"))


# RAG query function

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="chroma_db")
print(client.list_collections())


In [ ]:
!pip install groq


In [ ]:
import chromadb
from groq import Groq
from sentence_transformers import SentenceTransformer
import os
from dotenv import load_dotenv

# Load API keys
load_dotenv()

# Initialize Groq client
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Connect to ChromaDB
chroma_client = chromadb.PersistentClient(path="chroma_db")
collection = chroma_client.get_collection("hacking_knowledge")

# Load embedding model once
model = SentenceTransformer('all-MiniLM-L6-v2')

def rag_query(question, k=3):
    """
    question: user question
    k: number of similar chunks to fetch
    """

    # 1. Get embedding for question
    q_emb = model.encode(question).tolist()
    
    # 2. Search top-k chunks from Chroma
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=k
    )
    
    retrieved_docs = results["documents"][0]

    # 3. Combine context
    context = "\n\n".join(retrieved_docs)
    
    # 4. Create prompt
    prompt = f"""
    Answer the question using ONLY the context provided below.
    If the answer is not present in the context, reply:
    'The document does not contain this information.'

    CONTEXT:
    {context}

    QUESTION: {question}
    """

    # 5. Get LLM response from Groq
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    
    # FIXED line:
    answer = response.choices[0].message.content
    return answer


In [ ]:
rag_query("What is hacking?")


In [ ]:
rag_query("What is penetration testing?")

In [ ]:
# download_onnx.py
from huggingface_hub import hf_hub_download
import os

# where to save the model
out_dir = r"E:\RAG Project\models"
os.makedirs(out_dir, exist_ok=True)

# This tries to download a common ONNX artifact name used by many ONNX conversions.
# If this exact filename does not exist in the repo, hf_hub_download will raise an error;
# if that happens, use the manual browser method below.
try:
    onnx_path = hf_hub_download(
        repo_id="sentence-transformers/all-MiniLM-L6-v2",
        filename="onnx/model.onnx",  # common path used in ONNX-converted packages
        cache_dir=out_dir,
        repo_type="model",
    )
    print("Downloaded ONNX model to:", onnx_path)
except Exception as e:
    print("Automatic download failed:", e)
    print("If automatic fails, use the manual download method described in the instructions.")


In [ ]:
from huggingface_hub import hf_hub_download
import os

out_dir = r"E:\RAG Project\models"
os.makedirs(out_dir, exist_ok=True)

try:
    path = hf_hub_download(
        repo_id="sentence-transformers/all-mpnet-base-v2",
        filename="onnx/model.onnx",
        repo_type="model",
        cache_dir=out_dir
    )
    print("Downloaded ONNX model to:", path)
except Exception as e:
    print("FAILED:", e)


In [ ]:
!python download_mpnet_onnx.py
"E:\RAG Project\models\models--sentence-transformers--all-MiniLM-L6-v2\snapshots\c9745ed1d9f207416be6d2e6f8de32d1f16199bf\onnx\model.onnx"